In [ ]:
with open("response.json", "r") as f:
    text = f.read()
    response = json.loads(text)

In [ ]:
response['Blocks'][5]

{'BlockType': 'LINE',
 'Confidence': 94.36962127685547,
 'Text': 'Invoice to: :',
 'Geometry': {'BoundingBox': {'Width': 0.09464593976736069,
   'Height': 0.011991285718977451,
   'Left': 0.09286519885063171,
   'Top': 0.19513966143131256},
  'Polygon': [{'X': 0.09286519885063171, 'Y': 0.19514161348342896},
   {'X': 0.1875111311674118, 'Y': 0.19513966143131256},
   {'X': 0.187511146068573, 'Y': 0.20712900161743164},
   {'X': 0.0928652361035347, 'Y': 0.20713093876838684}]},
 'Id': '71518f70-5652-4f89-9885-857de1e9a322',
 'Relationships': [{'Type': 'CHILD',
   'Ids': ['971f3af3-902f-4174-ae30-0a4b6133123c',
    '859845be-fc7d-4ddf-b1e0-c614f5417266',
    '04ecb40d-ef03-4bf0-a06d-ff381fcf8038']}]}

In [ ]:
# Map blocks by Id
blocks = {b['Id']: b for b in response['Blocks']}

# Extract tables
tables = []

In [ ]:
for block in response['Blocks']:
    if block['BlockType'] == 'TABLE':
        table_data = []
        for rel in block.get('Relationships', []):
            if rel['Type'] == 'CHILD':
                for cell_id in rel['Ids']:
                    cell = blocks[cell_id]
                    if cell['BlockType'] == 'CELL':
                        row_idx = cell['RowIndex']
                        col_idx = cell['ColumnIndex']

                        text = ""
                        for rel2 in cell.get('Relationships', []):
                            if rel2['Type'] == 'CHILD':
                                for word_id in rel2['Ids']:
                                    word = blocks[word_id]
                                    if word['BlockType'] == 'WORD':
                                        text += word['Text'] + " "

                        table_data.append((row_idx, col_idx, text.strip()))
        tables.append(table_data)

# Convert each table to a Pandas DataFrame
dfs = []
for table in tables:
    df = pd.DataFrame(table, columns=["row", "col", "text"])
    df_pivot = df.pivot(index="row", columns="col", values="text")
    dfs.append(df_pivot)


In [ ]:
# Save tables to CSV
for i, df in enumerate(dfs, 1):
    df.to_csv(f"table_{i}.csv", index=False)
    print(f"Saved table_{i}.csv")

Saved table_1.csv
